[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_16/2_sql.ipynb)

# Day 16: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (32 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Day 1, day 7, day 30 | medium | 6 |
| Q2 | Monthly cohort table | medium | 6 |
| Q3 | Do gift-shop customers come back? | medium | 7 |
| Q4 | New, retained, resurrected, churned | medium | 8 |
| Q5 | Back the next day (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: UCI Online Retail (UK online shop, 2010-12 to 2011-12; CC BY 4.0).
def online_retail():
    path = os.path.join(DATA, "online_retail.csv.gz")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading Online Retail (about 23 MB, then converting the xlsx)")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://archive.ics.uci.edu/static/public/352/online+retail.zip").read()))
        pd.read_excel(io.BytesIO(z.read("Online Retail.xlsx"))).to_csv(path, index=False, compression="gzip")
    return pd.read_csv(path, dtype={"InvoiceNo": str, "StockCode": str})

_o = online_retail()
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id",
              "country"]
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o.to_sql("retail", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real table: UCI Online Retail** (a UK online gift shop, 2010-12-01 to 2011-12-09; CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (starts with 'C' = cancellation), `stock_code`, `description`, `quantity` (negative for returns), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price`, `customer_id` (NULL for about 25% of rows), `country` |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Day 1, day 7, day 30

*medium, about 6 min*

Shopping app (made up). **Day N retention** of a user = they have a `daily_activity` row exactly N days
after their `signup_date`. For each **signup week** (weeks start Monday: `date(d, 'weekday 0', '-6 days')`) return
`cohort_week`, `users`, `d1_pct`, `d7_pct`, `d30_pct` (percent of the cohort, 1 decimal).

Example: a user who signed up on 2024-01-03 counts for D7 if they were active on 2024-01-10, whatever they did on
other days.

Follow-ups: (a) Is D30 fair for users who signed up on 2024-03-20? (b) How is "rolling" (unbounded) D7 retention
different?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "active exactly N days after signup", grouped by cohort. Pattern: LEFT JOIN the activity table once per N with `activity_date = date(signup_date, '+N day')`, then average 0/1 flags.

</details>

<details><summary><b>Hint 2</b></summary>

1. FROM app_users u.
2. `LEFT JOIN daily_activity d1 ON d1.user_id = u.user_id AND d1.activity_date = date(u.signup_date, '+1 day')`, the same for 7 and 30.
3. Group by cohort week; `100.0 * AVG(CASE WHEN d1.user_id IS NOT NULL THEN 1 ELSE 0 END)`.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT date(u.signup_date, 'weekday 0', '-6 days') AS cohort_week,
       COUNT(*) AS users,
       ROUND(100.0 * AVG(CASE WHEN d1.user_id  IS NOT NULL THEN 1 ELSE 0 END), 1) AS d1_pct,
       ROUND(100.0 * AVG(CASE WHEN d7.user_id  IS NOT NULL THEN 1 ELSE 0 END), 1) AS d7_pct,
       ROUND(100.0 * AVG(CASE WHEN d30.user_id IS NOT NULL THEN 1 ELSE 0 END), 1) AS d30_pct
FROM app_users u
LEFT JOIN daily_activity d1  ON d1.user_id = u.user_id  AND d1.activity_date  = date(u.signup_date, '+1 day')
LEFT JOIN daily_activity d7  ON d7.user_id = u.user_id  AND d7.activity_date  = date(u.signup_date, '+7 day')
LEFT JOIN daily_activity d30 ON d30.user_id = u.user_id AND d30.activity_date = date(u.signup_date, '+30 day')
GROUP BY date(u.signup_date, 'weekday 0', '-6 days')
ORDER BY cohort_week
```

Result:

| cohort_week | users | d1_pct | d7_pct | d30_pct |
|---|---|---|---|---|
| 2024-01-01 | 174 | 45.4 | 29.9 | 18.4 |
| 2024-01-08 | 179 | 41.9 | 29.1 | 12.8 |
| 2024-01-15 | 170 | 49.4 | 26.5 | 14.7 |
| 2024-01-22 | 181 | 45.3 | 39.8 | 18.2 |
| 2024-01-29 | 175 | 50.9 | 28.6 | 14.3 |
| 2024-02-05 | 168 | 44.6 | 28.0 | 18.5 |
| 2024-02-12 | 200 | 41.5 | 28.5 | 17.5 |
| 2024-02-19 | 156 | 42.3 | 30.8 | 16.7 |
| 2024-02-26 | 97 | 39.2 | 22.7 | 12.4 |

**Why:** The date condition sits in the ON clause, so users without that activity row are kept (NULL) and count as 0. Since `daily_activity` has at most one row per user and day, the joins cannot multiply rows. Retention is about 40% to 50% on day 1, 25% to 30% on day 7 and 12% to 18% on day 30. (a) Here every signup is on or before Feb 29 and the data runs to Mar 31, so D30 is observable for everyone. In general, only include users with at least N days of history (`signup_date <= last_date - N`), otherwise recent cohorts look artificially bad. (b) Rolling D7 = active on day 7 **or later**; it is higher and never goes down as more data arrives. PostgreSQL: `u.signup_date + 7` or `+ INTERVAL '7 days'`.

**Complexity:** Three index lookups per user on (user_id, activity_date).

**Common mistakes:** Putting `d7.activity_date = ...` in WHERE (turns the LEFT JOIN into an inner join: D7 becomes 100%). Using `BETWEEN signup and signup + 7` (that is "active in the first week", a different metric). Not removing cohorts that are too young.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-left-join-nulls](https://mahsa-agz.github.io/ds-handbook/#sql-left-join-nulls), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q2. Monthly cohort table

*medium, about 6 min*

Shopping app. Build the classic cohort table: one row per **signup month** with `users` and the
percent of the cohort active in **month 0, 1 and 2** after signup (`m0`, `m1`, `m2`, 1 decimal). Month k = calendar
month of signup + k (a user who signed up on Jan 31 and came back on Feb 1 is active in month 1).

Months that are not observable yet (after March 2024) must be NULL, not 0.

Example shape: `2024-01 | 779 | 100.0 | 58.3 | 38.3`.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "cohort by first month, activity by months since". Pattern: month index = year * 12 + month, the difference gives k; then COUNT(DISTINCT CASE WHEN k = ... ) / cohort size.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE act: DISTINCT user, cohort month, k = month index of activity minus month index of signup.
2. CTE size: users per cohort.
3. Join and pivot with `COUNT(DISTINCT CASE WHEN k = 1 THEN user_id END)`. Wrap m2 in a CASE that returns NULL when cohort month + 2 is after the last data month.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH act AS (
    SELECT DISTINCT d.user_id,
           substr(u.signup_date, 1, 7) AS cohort,
           (CAST(substr(d.activity_date, 1, 4) AS INTEGER) * 12 + CAST(substr(d.activity_date, 6, 2) AS INTEGER))
         - (CAST(substr(u.signup_date, 1, 4) AS INTEGER) * 12 + CAST(substr(u.signup_date, 6, 2) AS INTEGER)) AS k
    FROM daily_activity d
    JOIN app_users u ON u.user_id = d.user_id
), size AS (
    SELECT substr(signup_date, 1, 7) AS cohort, COUNT(*) AS users
    FROM app_users
    GROUP BY substr(signup_date, 1, 7)
), last AS (
    SELECT MAX(substr(activity_date, 1, 7)) AS last_month FROM daily_activity
)
SELECT s.cohort, s.users,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.k = 0 THEN a.user_id END) / s.users, 1) AS m0,
       CASE WHEN date(s.cohort || '-01', '+1 month') <= l.last_month || '-01'
            THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.k = 1 THEN a.user_id END) / s.users, 1) END AS m1,
       CASE WHEN date(s.cohort || '-01', '+2 month') <= l.last_month || '-01'
            THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN a.k = 2 THEN a.user_id END) / s.users, 1) END AS m2
FROM size s
JOIN act a ON a.cohort = s.cohort
CROSS JOIN last l
GROUP BY s.cohort, s.users, l.last_month
ORDER BY s.cohort
```

Result:

| cohort | users | m0 | m1 | m2 |
|---|---|---|---|---|
| 2024-01 | 779 | 100.0 | 58.3 | 38.3 |
| 2024-02 | 721 | 100.0 | 58.8 | NULL |

**Why:** The month index turns calendar months into integers, so 'months since signup' is a subtraction that works across years. Conditional COUNT(DISTINCT) pivots k into columns. m0 is 100% because every user is active on signup day. The February cohort's month 2 is April, which is not in the data, so it must be NULL: a 0 would read as "everybody churned". PostgreSQL: `(EXTRACT(YEAR FROM age(date_trunc('month', activity_date), date_trunc('month', signup_date))) * 12 + EXTRACT(MONTH FROM age(...)))`, or simply compare `date_trunc('month', ...)` values.

**Complexity:** One pass over daily_activity with a join to users, then a small group by.

**Common mistakes:** Dividing by the number of active users in month 0 instead of the cohort size (same here, different when users can sign up without being active). Days / 30 instead of calendar months. Showing 0 for unobserved cells.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-pivot](https://mahsa-agz.github.io/ds-handbook/#sql-pivot), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when)

</details>

---
### Q3. Do gift-shop customers come back?

*medium, about 7 min*

Online Retail (real). A customer's **cohort** is the month of their first purchase (ignore missing
`customer_id` and cancellation invoices starting with 'C'). For each cohort return `cohort` ('YYYY-MM'),
`customers` and the percent who bought again in month +1, +2 and +3 (`m1`, `m2`, `m3`, 1 decimal). Cells after the
last month of data must be NULL.

Then name two reasons why the 2010-12 cohort looks so much better than the others.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: same cohort template as Q2, but the cohort is the first purchase month (MIN per customer) instead of a signup date.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE buys: DISTINCT customer and month index (`year * 12 + month`).
2. CTE firsts: MIN month index per customer.
3. k = month index minus cohort index; pivot with COUNT(DISTINCT CASE ...); NULL when cohort + k is after the last month index. `printf('%d-%02d', ...)` turns the index back into text.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH buys AS (
    SELECT DISTINCT customer_id,
           CAST(substr(invoice_date, 1, 4) AS INTEGER) * 12 + CAST(substr(invoice_date, 6, 2) AS INTEGER) AS mi
    FROM retail
    WHERE customer_id IS NOT NULL AND invoice_no NOT LIKE 'C%'
), firsts AS (
    SELECT customer_id, MIN(mi) AS cohort_mi FROM buys GROUP BY customer_id
), joined AS (
    SELECT f.cohort_mi, b.mi - f.cohort_mi AS k, b.customer_id
    FROM buys b
    JOIN firsts f ON f.customer_id = b.customer_id
), last AS (
    SELECT MAX(mi) AS last_mi FROM buys
)
SELECT printf('%d-%02d', (cohort_mi - 1) / 12, (cohort_mi - 1) % 12 + 1) AS cohort,
       COUNT(DISTINCT CASE WHEN k = 0 THEN customer_id END) AS customers,
       CASE WHEN cohort_mi + 1 <= last_mi THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN k = 1 THEN customer_id END)
            / COUNT(DISTINCT CASE WHEN k = 0 THEN customer_id END), 1) END AS m1,
       CASE WHEN cohort_mi + 2 <= last_mi THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN k = 2 THEN customer_id END)
            / COUNT(DISTINCT CASE WHEN k = 0 THEN customer_id END), 1) END AS m2,
       CASE WHEN cohort_mi + 3 <= last_mi THEN ROUND(100.0 * COUNT(DISTINCT CASE WHEN k = 3 THEN customer_id END)
            / COUNT(DISTINCT CASE WHEN k = 0 THEN customer_id END), 1) END AS m3
FROM joined
CROSS JOIN last
GROUP BY cohort_mi, last_mi
ORDER BY cohort_mi
```

Result:

| cohort | customers | m1 | m2 | m3 |
|---|---|---|---|---|
| 2010-12 | 885 | 36.6 | 32.3 | 38.4 |
| 2011-01 | 417 | 22.1 | 26.6 | 23.0 |
| 2011-02 | 380 | 18.7 | 18.7 | 28.4 |
| 2011-03 | 452 | 15.0 | 25.2 | 19.9 |
| 2011-04 | 300 | 21.3 | 20.3 | 21.0 |
| 2011-05 | 284 | 19.0 | 17.3 | 17.3 |
| 2011-06 | 242 | 17.4 | 15.7 | 26.4 |
| 2011-07 | 188 | 18.1 | 20.7 | 22.3 |
| 2011-08 | 169 | 20.7 | 24.9 | 24.3 |
| 2011-09 | 299 | 23.4 | 30.1 | 11.4 |
| 2011-10 | 358 | 24.0 | 11.5 | NULL |
| 2011-11 | 324 | 11.1 | NULL | NULL |

(13 rows, first 12 shown)

**Why:** Same cohort logic on real data. The 2010-12 cohort keeps 32% to 38% while later cohorts keep about 15% to 25%. Reasons: (1) left censoring: the shop existed before the data starts, so 2010-12 "new" customers include all loyal existing customers seen for the first time; (2) many customers are wholesalers who buy every month. Also note right censoring at the end: 2011-12 has only 9 days, so the last observed cell of each cohort (for example 2011-09 m3 = 11.4) is too low. Exclude or flag partial months.

**Complexity:** DISTINCT over about 400k lines, then joins and a group by on about 13k customer-months.

**Common mistakes:** Counting rows instead of distinct customers. Using the first invoice in the whole table as "new" without thinking about data that starts mid-life. Letting the partial last month look like a real drop.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning)

</details>

---
### Q4. New, retained, resurrected, churned

*medium, about 8 min*

Shopping app. Monthly **growth accounting**. A user is active in a month if they have any
`daily_activity` row in it. For each month M:
- `new_users`: active in M, and M is their first active month;
- `retained`: active in M and in M - 1;
- `resurrected`: active in M, not in M - 1, but active at some point before;
- `churned`: active in M - 1 but not in M;
- `mau`: active in M (= new + retained + resurrected);
- `quick_ratio` = (new + resurrected) / churned (2 decimals; NULL when churned is 0).

Example: a user active in January and March, but not February, is churned in February and resurrected in March.

What does a quick ratio below 1 tell the PM?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: compare each user's month with the previous / next month. Pattern: a (user, month) table self joined to itself on the previous month (LEFT JOIN, IS NULL means "not active then").

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE m: DISTINCT user_id, month. CTE firsts: MIN(month) per user.
2. Active side: m cur LEFT JOIN m prev ON same user AND prev.month = month before cur.month; CASE counts per month.
3. Churn side: m prev LEFT JOIN m nxt ON the next month; rows where nxt IS NULL, grouped by the next month. 4. Join both on month. Month arithmetic: `strftime('%Y-%m', date(month || '-01', '-1 month'))`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH m AS (
    SELECT DISTINCT user_id, substr(activity_date, 1, 7) AS month FROM daily_activity
), firsts AS (
    SELECT user_id, MIN(month) AS first_month FROM m GROUP BY user_id
), active AS (
    SELECT cur.month,
           COUNT(*) AS mau,
           SUM(CASE WHEN cur.month = f.first_month THEN 1 ELSE 0 END) AS new_users,
           SUM(CASE WHEN prev.user_id IS NOT NULL THEN 1 ELSE 0 END) AS retained,
           SUM(CASE WHEN prev.user_id IS NULL AND cur.month > f.first_month THEN 1 ELSE 0 END) AS resurrected
    FROM m cur
    JOIN firsts f ON f.user_id = cur.user_id
    LEFT JOIN m prev ON prev.user_id = cur.user_id
                    AND prev.month = strftime('%Y-%m', date(cur.month || '-01', '-1 month'))
    GROUP BY cur.month
), churn AS (
    SELECT strftime('%Y-%m', date(prev.month || '-01', '+1 month')) AS month, COUNT(*) AS churned
    FROM m prev
    LEFT JOIN m nxt ON nxt.user_id = prev.user_id
                   AND nxt.month = strftime('%Y-%m', date(prev.month || '-01', '+1 month'))
    WHERE nxt.user_id IS NULL
    GROUP BY strftime('%Y-%m', date(prev.month || '-01', '+1 month'))
)
SELECT a.month, a.mau, a.new_users, a.retained, a.resurrected,
       COALESCE(c.churned, 0) AS churned,
       ROUND(1.0 * (a.new_users + a.resurrected) / NULLIF(c.churned, 0), 2) AS quick_ratio
FROM active a
LEFT JOIN churn c ON c.month = a.month
ORDER BY a.month
```

Result:

| month | mau | new_users | retained | resurrected | churned | quick_ratio |
|---|---|---|---|---|---|---|
| 2024-01 | 779 | 779 | 0 | 0 | 0 | NULL |
| 2024-02 | 1175 | 721 | 454 | 0 | 325 | 2.22 |
| 2024-03 | 722 | 0 | 644 | 78 | 531 | 0.15 |

**Why:** Every active user-month falls in exactly one of new / retained / resurrected, so they add up to MAU (a good check: 644 + 78 = 722 in March). Churn lives on the other side (users who are missing), so it needs its own anti join (LEFT JOIN ... IS NULL). In March no new users arrive (signups ended in February), 531 users churn and the quick ratio is 0.15: the product is shrinking, because far more users leave than come in or come back. Above 1 means growth; healthy consumer apps aim well above 1. The churn CTE also produces a row for April (churned from March); the LEFT JOIN from active months drops it.

**Complexity:** A few passes over the user-month table (about 2,700 rows) with index-like joins on user and month.

**Common mistakes:** Counting churn from the current month (impossible: churned users have no row in M). Treating a returning user as new. Forgetting NULLIF and dividing by 0 in January.

**Learn more:** [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention), [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [stats-product-metrics](https://mahsa-agz.github.io/ds-handbook/#stats-product-metrics)

</details>

---
### Q5. Back the next day (review)

*medium, about 5 min*

Shopping app. For each day from **2024-03-01 to 2024-03-30**, return `activity_date`, `dau`,
`back_next_day` (how many of that day's active users are also active the next day) and `pct_back` (1 decimal).

Example: if 260 users are active on Mar 1 and 195 of them are active on Mar 2, pct_back is 75.0.

Why does the question stop at March 30?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Pattern: self join of daily_activity to itself: same user, date + 1 day; LEFT JOIN so the denominator keeps everybody.

</details>

<details><summary><b>Hint 2</b></summary>

1. `daily_activity a LEFT JOIN daily_activity b ON b.user_id = a.user_id AND b.activity_date = date(a.activity_date, '+1 day')`.
2. Group by a.activity_date: COUNT(*) for dau, COUNT(b.user_id) for back_next_day.

</details>

<details><summary><b>Solution</b></summary>

```sql
SELECT a.activity_date,
       COUNT(*)          AS dau,
       COUNT(b.user_id)  AS back_next_day,
       ROUND(100.0 * COUNT(b.user_id) / COUNT(*), 1) AS pct_back
FROM daily_activity a
LEFT JOIN daily_activity b
       ON b.user_id = a.user_id
      AND b.activity_date = date(a.activity_date, '+1 day')
WHERE a.activity_date >= '2024-03-01' AND a.activity_date <= '2024-03-30'
GROUP BY a.activity_date
ORDER BY a.activity_date
```

Result:

| activity_date | dau | back_next_day | pct_back |
|---|---|---|---|
| 2024-03-01 | 260 | 195 | 75.0 |
| 2024-03-02 | 256 | 185 | 72.3 |
| 2024-03-03 | 253 | 185 | 73.1 |
| 2024-03-04 | 236 | 179 | 75.8 |
| 2024-03-05 | 239 | 169 | 70.7 |
| 2024-03-06 | 227 | 158 | 69.6 |
| 2024-03-07 | 216 | 157 | 72.7 |
| 2024-03-08 | 214 | 144 | 67.3 |
| 2024-03-09 | 194 | 145 | 74.7 |
| 2024-03-10 | 197 | 139 | 70.6 |
| 2024-03-11 | 191 | 141 | 73.8 |
| 2024-03-12 | 198 | 137 | 69.2 |

(30 rows, first 12 shown)

**Why:** The self join pairs each user-day with the same user's next day. COUNT(*) counts all rows of `a` (the LEFT JOIN keeps them) and COUNT(b.user_id) only the matched ones. Day-over-day return is about 65% to 76%. March 31 is the last day of data, so its "next day" is unknown: including it would show a fake 0%. The same logic with LEAD is possible, but the self join reads more directly as "exists on date + 1".

**Complexity:** One lookup per user-day on (user_id, activity_date).

**Common mistakes:** Inner join (dau becomes back_next_day). Joining on date only, not on user. Including the last day of data.

**Learn more:** [sql-self-join](https://mahsa-agz.github.io/ds-handbook/#sql-self-join), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates), [sql-retention](https://mahsa-agz.github.io/ds-handbook/#sql-retention)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_16/3_stats.ipynb)